In [ ]:
import numpy as np
import pandas as pd
import os
import time

# --- 1. LOCAL DIRECTORY SETUP ---
# Gets the current working directory (where this script is saved)
current_dir = os.getcwd()
base_path = os.path.join(current_dir, 'Full Comparison')

# Create the folder if it doesn't already exist
if not os.path.exists(base_path):
    os.makedirs(base_path)
    print(f"Created directory: {base_path}")
else:
    print(f"Saving to existing directory: {base_path}")

# --- 2. GLOBAL CONFIGURATION ---
T = 100             
prices = np.arange(0, 21.5, 0.5) 
num_trials = 20     
num_sims = 10000    

splits = [0.20, 0.50]
modes = ['uniform_baseline', 'linear', 'exp_gentle', 'exp_steep', 'concave']
inv_multipliers = [0.8, 0.7, 0.6]

# Define policy names for file output
policy_names = {
    'Smart_Sim': 'Smart_DP',
    'Dyn_TwoFare_Lazy_5_Sim': 'Lazy_Dyn_TwoFare_5',
    'Dyn_TwoFare_Lazy_10_Sim': 'Lazy_Dyn_TwoFare_10',
    'TwoFare_Theory': 'Theoretical_TwoFare',
    'TwoFare_Sim': 'Simulated_TwoFare',
    'Dumb_Sim': 'Dumb_Static'
}

# --- FUNCTIONS ---

def generate_gamma(T, mode='uniform_baseline'):
    t_values = np.arange(T)
    norm_t = t_values / (T - 1) 
    
    if mode == 'uniform_baseline':
        return np.random.uniform(0.15, 0.35, T)
    elif mode == 'linear':
        means = 0.1 + 0.3 * norm_t
    elif mode == 'exp_gentle':
        b = np.log(4) 
        means = 0.1 * np.exp(b * norm_t)
    elif mode == 'exp_steep':
        k = 5.0
        means = 0.1 + 0.3 * ((np.exp(k * norm_t) - 1) / (np.exp(k) - 1))
    elif mode == 'concave':
        k = 5.0
        means = 0.1 + 0.3 * ((1 - np.exp(-k * norm_t)) / (1 - np.exp(-k)))
    else:
        raise ValueError(f"Unknown mode: {mode}")

    lower_bounds = np.maximum(0.001, means - 0.1)
    upper_bounds = means + 0.1
    return np.random.uniform(lower_bounds, upper_bounds)

def solve_smart_dp(T, C, alpha, gamma, prices):
    V = np.zeros((T + 1, C + 1))
    p_star = np.zeros((T + 1, C + 1))
    for t in range(T - 1, -1, -1):
        lambda_val = np.exp(-gamma[t] * prices)
        for x in range(1, C + 1):
            prob_buy = alpha[t] * lambda_val
            expected_values = prob_buy * (prices + V[t+1, x-1]) + (1 - prob_buy) * V[t+1, x]
            best_idx = np.argmax(expected_values)
            V[t, x] = expected_values[best_idx]
            p_star[t, x] = prices[best_idx]
    return V, p_star

def find_best_static_price(T, C, alpha, gamma, prices):
    best_price, max_revenue = 0, -1.0
    for p in prices:
        V_static = np.zeros((T + 1, C + 1))
        for t in range(T - 1, -1, -1):
            lam = np.exp(-gamma[t] * p)
            prob_buy = alpha[t] * lam
            V_static[t, 1:] = prob_buy * (p + V_static[t+1, :-1]) + (1 - prob_buy) * V_static[t+1, 1:]
        if V_static[0, C] > max_revenue:
            max_revenue = V_static[0, C]
            best_price = p
    return best_price, max_revenue

def solve_theoretical_two_fare(T, C, alpha, gamma, prices, split_ratio):
    div_point = int(np.round(T * split_ratio))
    best_p1, best_p2, max_rev = 0, 0, -1.0
    for p1 in prices:
        theta_1 = np.sum(alpha[:div_point] * np.exp(-gamma[:div_point] * p1))
        for p2 in prices:
            theta_2 = np.sum(alpha[div_point:] * np.exp(-gamma[div_point:] * p2))
            sales_1 = min(theta_1, C)
            sales_2 = min(C - sales_1, theta_2)
            expected_rev = (p1 * sales_1) + (p2 * sales_2)
            if expected_rev > max_rev:
                max_rev, best_p1, best_p2 = expected_rev, p1, p2
    return best_p1, best_p2, max_rev

def get_or_compute_p1(t, x, T, alpha, gamma, prices, split_ratio, p1_table):
    if not np.isnan(p1_table[t, x]):
        return p1_table[t, x]
    
    remaining_T = T - t
    div_point = int(np.round(remaining_T * split_ratio))
    alpha_sub, gamma_sub = alpha[t:], gamma[t:]
    
    best_p1, max_rev = 0, -1.0
    for p1 in prices:
        theta_1 = np.sum(alpha_sub[:div_point] * np.exp(-gamma_sub[:div_point] * p1))
        for p2 in prices:
            theta_2 = np.sum(alpha_sub[div_point:] * np.exp(-gamma_sub[div_point:] * p2))
            sales_1 = min(theta_1, x)
            sales_2 = min(x - sales_1, theta_2)
            expected_rev = (p1 * sales_1) + (p2 * sales_2)
            if expected_rev > max_rev:
                max_rev, best_p1 = expected_rev, p1

    p1_table[t, x] = best_p1
    return best_p1

def simulate_revenue(T, C, alpha, gamma, trial_seed, prices=None, split_ratio=None, 
                     p_star=None, static_price=None, two_fare_prices=None, 
                     dynamic_two_fare_table=None, recompute_interval=None, runs=10000):
    total_revenue = 0.0
    div_point = int(np.round(T * split_ratio)) if split_ratio else 0

    for r in range(runs):
        # CRN: Force identical customer behavior for this specific run across all policies
        np.random.seed((trial_seed * 100000) + r)
        
        current_rev, x, active_price = 0, C, 0
        for t in range(T):
            if x == 0: break

            if p_star is not None:
                active_price = p_star[t, x]
            elif dynamic_two_fare_table is not None:
                if t % recompute_interval == 0:
                    active_price = get_or_compute_p1(t, x, T, alpha, gamma, prices, split_ratio, dynamic_two_fare_table)
            elif two_fare_prices is not None:
                active_price = two_fare_prices[0] if t < div_point else two_fare_prices[1]
            else:
                active_price = static_price

            if np.random.uniform(0, 1) <= alpha[t]:
                if np.random.uniform(0, 1) <= np.exp(-gamma[t] * active_price):
                    current_rev += active_price
                    x -= 1
        total_revenue += current_rev

    return total_revenue / runs

# --- MAIN EXECUTION ---
all_results = []

print(f"{'='*80}\nSTARTING MASSIVE GRID SEARCH\n{'='*80}")

for split in splits:
    print(f"\n>>> PROCESSING SPLIT: {split*100}-{(1-split)*100} <<<")
    
    for mode in modes:
        for inv_mult in inv_multipliers:
            print(f"  Mode: {mode.upper():<16} | Inv: {inv_mult}")
            
            for trial in range(1, num_trials + 1):
                # 1. Set seed for generating the environment
                # trial_seed = int(split * 100) + hash(mode) % 10000 + int(inv_mult * 10) + trial
                # np.random.seed(trial_seed)
                
                # 1. Set seed for generating the environment (1 to 20)
                trial_seed = trial
                np.random.seed(trial_seed)
                
                alpha = np.random.uniform(0.5, 1.0, T)
                gamma = generate_gamma(T, mode=mode)
                
                L = 0.3 * np.sum(alpha) # Theoretical expected demand
                C = int(inv_mult * L)
                
                # 2. Solvers & Pre-computations
                V, p_star = solve_smart_dp(T, C, alpha, gamma, prices)
                best_static_p, _ = find_best_static_price(T, C, alpha, gamma, prices)
                best_p1, best_p2, theoretical_two_fare_rev = solve_theoretical_two_fare(T, C, alpha, gamma, prices, split)
                
                lazy_table_5 = np.full((T, C + 1), np.nan)
                lazy_table_10 = np.full((T, C + 1), np.nan)

                # 3. Simulations (Passing trial_seed for CRN)
                sim_smart = simulate_revenue(T, C, alpha, gamma, trial_seed, p_star=p_star, runs=num_sims)
                sim_dumb = simulate_revenue(T, C, alpha, gamma, trial_seed, static_price=best_static_p, runs=num_sims)
                sim_two_fare = simulate_revenue(T, C, alpha, gamma, trial_seed, two_fare_prices=(best_p1, best_p2), split_ratio=split, runs=num_sims)
                sim_lazy_5 = simulate_revenue(T, C, alpha, gamma, trial_seed, prices=prices, split_ratio=split, dynamic_two_fare_table=lazy_table_5, recompute_interval=5, runs=num_sims)
                sim_lazy_10 = simulate_revenue(T, C, alpha, gamma, trial_seed, prices=prices, split_ratio=split, dynamic_two_fare_table=lazy_table_10, recompute_interval=10, runs=num_sims)
                
                # 4. Save Record
                all_results.append({
                    'Split': split, 'Mode': mode, 'Inv_Mult': inv_mult, 'Trial': trial,
                    'Smart_Sim': sim_smart,
                    'Dyn_TwoFare_Lazy_5_Sim': sim_lazy_5,
                    'Dyn_TwoFare_Lazy_10_Sim': sim_lazy_10,
                    'TwoFare_Theory': theoretical_two_fare_rev,
                    'TwoFare_Sim': sim_two_fare,
                    'Dumb_Sim': sim_dumb
                })

# --- DATA EXPORT ARCHITECTURE ---
'''
print(f"\n{'='*80}\nSAVING WORKBOOKS\n{'='*80}")
df_all = pd.DataFrame(all_results)

# 1. The 12 Policy Workbooks
for split in splits:
    df_split = df_all[df_all['Split'] == split]
    
    for policy_col, filename_tag in policy_names.items():
        wb_name = os.path.join(base_path, f'{filename_tag}_Split_{int(split*100)}_{int((1-split)*100)}.xlsx')
        
        with pd.ExcelWriter(wb_name, engine='openpyxl') as writer:
            for mode in modes:
                for inv in inv_multipliers:
                    sheet_name = f"{mode[:10]}_Inv{inv}" # Limit sheet name length
                    
                    df_combo = df_split[(df_split['Mode'] == mode) & (df_split['Inv_Mult'] == inv)]
                    df_out = df_combo[['Trial', policy_col]].copy()
                    df_out.to_excel(writer, sheet_name=sheet_name, index=False)
                    
        print(f"Saved: {wb_name}")

# 2. The Master Summary Workbook
master_wb_name = os.path.join(base_path, 'Master_Summary_All_Comparisons.xlsx')

with pd.ExcelWriter(master_wb_name, engine='openpyxl') as writer:
    for split in splits:
        df_split = df_all[df_all['Split'] == split]
        
        summary = df_split.groupby(['Mode', 'Inv_Mult'])[list(policy_names.keys())].mean().reset_index()
        
        dumb_col = 'Dumb_Sim'
        summary['% Imp Smart'] = (summary['Smart_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp Lazy_5'] = (summary['Dyn_TwoFare_Lazy_5_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp Lazy_10'] = (summary['Dyn_TwoFare_Lazy_10_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp 2Fare_Theory'] = (summary['TwoFare_Theory'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp 2Fare_Sim'] = (summary['TwoFare_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        
        sheet_name = f"Split_{int(split*100)}_{int((1-split)*100)}"
        summary.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved Master Summary: {master_wb_name}")
print("EXPERIMENT COMPLETE.")
'''

# --- DATA EXPORT ARCHITECTURE ---
print(f"\n{'='*80}\nSAVING WORKBOOKS\n{'='*80}")
df_all = pd.DataFrame(all_results)

# --- NEW: Create a dedicated subfolder for the Seed 1-20 data ---
export_folder = os.path.join(base_path, 'Seeds_1_to_20')
if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created dedicated export directory: {export_folder}\n")

# 1. The 12 Policy Workbooks
for split in splits:
    df_split = df_all[df_all['Split'] == split]
    
    for policy_col, filename_tag in policy_names.items():
        # Changed base_path to export_folder
        wb_name = os.path.join(export_folder, f'{filename_tag}_Split_{int(split*100)}_{int((1-split)*100)}.xlsx')
        
        with pd.ExcelWriter(wb_name, engine='openpyxl') as writer:
            for mode in modes:
                for inv in inv_multipliers:
                    sheet_name = f"{mode[:10]}_Inv{inv}" # Limit sheet name length
                    
                    df_combo = df_split[(df_split['Mode'] == mode) & (df_split['Inv_Mult'] == inv)]
                    df_out = df_combo[['Trial', policy_col]].copy()
                    df_out.to_excel(writer, sheet_name=sheet_name, index=False)
                    
        print(f"Saved: {wb_name}")

# 2. The Master Summary Workbook
# Changed base_path to export_folder
master_wb_name = os.path.join(export_folder, 'Master_Summary_All_Comparisons.xlsx')

with pd.ExcelWriter(master_wb_name, engine='openpyxl') as writer:
    for split in splits:
        df_split = df_all[df_all['Split'] == split]
        
        summary = df_split.groupby(['Mode', 'Inv_Mult'])[list(policy_names.keys())].mean().reset_index()
        
        dumb_col = 'Dumb_Sim'
        summary['% Imp Smart'] = (summary['Smart_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp Lazy_5'] = (summary['Dyn_TwoFare_Lazy_5_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp Lazy_10'] = (summary['Dyn_TwoFare_Lazy_10_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp 2Fare_Theory'] = (summary['TwoFare_Theory'] - summary[dumb_col]) / summary[dumb_col] * 100
        summary['% Imp 2Fare_Sim'] = (summary['TwoFare_Sim'] - summary[dumb_col]) / summary[dumb_col] * 100
        
        sheet_name = f"Split_{int(split*100)}_{int((1-split)*100)}"
        summary.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved Master Summary: {master_wb_name}")
print("EXPERIMENT COMPLETE.")

Saving to existing directory: /Users/alexandramagnusson/Downloads/Cornell/Spring 2026/Independent Study/Full Comparison/Full Comparison
STARTING MASSIVE GRID SEARCH

>>> PROCESSING SPLIT: 20.0-80.0 <<<
  Mode: UNIFORM_BASELINE | Inv: 0.8
  Mode: UNIFORM_BASELINE | Inv: 0.7
  Mode: UNIFORM_BASELINE | Inv: 0.6
  Mode: LINEAR           | Inv: 0.8
  Mode: LINEAR           | Inv: 0.7
  Mode: LINEAR           | Inv: 0.6
  Mode: EXP_GENTLE       | Inv: 0.8
